# CPU Training With TrainLens

This notebook trains a tiny CPU-only classifier with pure Python, stores the dataset notes, training parameters, metrics, and traces in memory, and then asks TrainLens to explain the result.

The LLM provider is configured for LLM7 through the OpenAI-compatible API. The API key is requested with `getpass` so it is not saved in the notebook.

## 1. Install TrainLens

Run this cell in a clean notebook environment. If you are developing TrainLens locally, replace it with `python -m pip install -e ..` from the repository root.

In [ ]:
%pip install -q trainlens

## 2. Configure The LLM Provider

Set your LLM7 API key when prompted. The key is kept in the current Python process only.

In [ ]:
import getpass
import os

import trainlens

os.environ["TRAINLENS_LLM_BASE_URL"] = "https://api.llm7.io/v1"
os.environ["TRAINLENS_LLM_MODEL"] = "gpt-5.4"
os.environ["TRAINLENS_LLM_TIMEOUT_SECONDS"] = "180"

if not os.environ.get("TRAINLENS_LLM_API_KEY"):
    os.environ["TRAINLENS_LLM_API_KEY"] = getpass.getpass("LLM7 API key: ")

trainlens.__version__

## 3. Train A Tiny CPU Model

The model is logistic regression trained with gradient descent on a synthetic, linearly separable binary classification dataset. It uses only the Python standard library.

In [ ]:
import math
import random

random.seed(7)

dataset_name = "synthetic_cpu_binary_classification"
dataset_notes = (
    "240 synthetic 2D samples; binary labels; validation split is balanced; "
    "generated in memory for a CPU smoke test."
)
model_name = "pure-python-logistic-regression"
hardware_notes = "CPU-only example; no GPU, torch, sklearn, or external dataset required."
training_params = {
    "epochs": 30,
    "learning_rate": 0.45,
    "train_samples": 180,
    "validation_samples": 60,
    "optimizer": "manual gradient descent",
}


def sigmoid(value: float) -> float:
    return 1.0 / (1.0 + math.exp(-value))


def make_dataset(size: int = 240) -> list[tuple[float, float, int]]:
    rows = []
    for _ in range(size):
        x1 = random.uniform(-2.0, 2.0)
        x2 = random.uniform(-2.0, 2.0)
        margin = 1.25 * x1 - 0.85 * x2 + random.gauss(0.0, 0.35)
        label = 1 if margin > 0 else 0
        rows.append((x1, x2, label))
    return rows


def evaluate(rows: list[tuple[float, float, int]], weights: list[float]) -> tuple[float, float]:
    total_loss = 0.0
    correct = 0
    for x1, x2, label in rows:
        prediction = sigmoid(weights[0] + weights[1] * x1 + weights[2] * x2)
        prediction = min(max(prediction, 1e-7), 1.0 - 1e-7)
        total_loss += -(label * math.log(prediction) + (1 - label) * math.log(1 - prediction))
        correct += int((prediction >= 0.5) == bool(label))
    return total_loss / len(rows), correct / len(rows)


dataset = make_dataset()
train_rows = dataset[:180]
validation_rows = dataset[180:]
weights = [0.0, 0.0, 0.0]
history = {"train_loss": [], "eval_loss": [], "accuracy": [], "val_accuracy": []}
training_trace = []

for epoch in range(1, training_params["epochs"] + 1):
    gradients = [0.0, 0.0, 0.0]
    for x1, x2, label in train_rows:
        prediction = sigmoid(weights[0] + weights[1] * x1 + weights[2] * x2)
        error = prediction - label
        gradients[0] += error
        gradients[1] += error * x1
        gradients[2] += error * x2

    learning_rate = training_params["learning_rate"]
    for index in range(len(weights)):
        weights[index] -= learning_rate * gradients[index] / len(train_rows)

    train_loss, train_accuracy = evaluate(train_rows, weights)
    validation_loss, validation_accuracy = evaluate(validation_rows, weights)
    history["train_loss"].append(round(train_loss, 4))
    history["eval_loss"].append(round(validation_loss, 4))
    history["accuracy"].append(round(train_accuracy, 4))
    history["val_accuracy"].append(round(validation_accuracy, 4))
    training_trace.append(
        {
            "epoch": epoch,
            "train_loss": round(train_loss, 4),
            "eval_loss": round(validation_loss, 4),
            "accuracy": round(train_accuracy, 4),
            "val_accuracy": round(validation_accuracy, 4),
        }
    )

history

## 4. Explain The Training Run

TrainLens reads the variables above from notebook memory and sends structured evidence to the configured LLM provider.

In [ ]:
%load_ext trainlens.magic.extension
%explain_training